# GNN-BERT Music Context: End-to-End Demo

Runs one inference example through all four tasks using the offline synthetic
data generator (`src/synthetic_data.py`), so this notebook works without
internet access or real datasets. Swap `sd.make_dataset(...)` for a real
loader built on `audio_features.py` + `graph_builder.py` once you have data
in `data/raw/`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import torch
import synthetic_data as sd
import train as T

cfg = T.load_config('../config.yaml')
T.set_seed(42)
dataset = sd.make_dataset(n=120, num_tags=cfg['data']['num_tags'])
split = sd.make_split(dataset)
print({k: len(v) for k, v in split.items()})

## Task 1: BERT tag classifier

In [ ]:
class Args:
    epochs = 2
    batch_size = 16
    graph = 'segment'

bert_model, bert_history = T.train_task1(cfg, Args(), split)

## Task 2: GNN on segment/chord graphs

In [ ]:
gnn_model, gnn_history = T.train_task2(cfg, Args(), split)

## Task 3: GNN-BERT cross-attention fusion (+ emotion regression)

In [ ]:
fusion_model, fusion_history = T.train_task3(cfg, Args(), split)

## Task 4: Contrastive dual-encoder + retrieval

In [ ]:
dual_model, dual_history = T.train_task4(cfg, Args(), split)
T.save_qualitative_retrieval(dual_model, split['test'], 'segment', '../results/retrieval_examples')

## One end-to-end inference example (Task 3 fusion model)

In [ ]:
example = split['test'][0]
g = T.graph_batch([example], 'segment')
ids, mask = T.text_tensor([example])
with torch.no_grad():
    out = fusion_model(g.x, g.edge_index, g.batch, ids, mask)
probs = torch.sigmoid(out['tag_logits'])[0]
top_tags = torch.topk(probs, k=5).indices.tolist()
print('track:', example['track_id'])
print('predicted top-5 tag indices:', top_tags)
print('predicted valence/arousal:', out['valence'].item(), out['arousal'].item())
print('true valence/arousal:', example['valence'], example['arousal'])